# Phase 3 — Build the model and prove it can learn

**Goal:** build the inverse dynamics model, watch a real clip flow through it layer by layer, and run two sanity checks before spending an hour on real training:

1. **The untrained model is a fair coin.** Its loss should be about ln(9) ≈ 2.20.
2. **It can memorize one small batch.** If it can't learn 18 clips by heart, it can't learn 70,000, and something is broken.

### A neural network in four sentences

A model is a function with millions of adjustable numbers (**weights**); ours has about 7.9 million. For each clip it outputs 9 scores (**logits**), one per action; higher means "more likely." The **loss** measures how wrong the scores are: low when the right action gets the highest score, high otherwise. **Training** repeatedly nudges every weight a tiny bit in the direction that lowers the loss (gradient descent). That's the same idea as a controller driving an error signal to zero.

The code lives in `src/model.py` and `src/data.py`. Open both; they're short and commented. This notebook uses them.

## 1. Setup

In [1]:
%pip install -q torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, time, math
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "README.md")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
OUT_DIR = os.path.join(ROOT, "data", "processed")

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from src.data import ClipDataset
from src.model import IDM9Way, IDMFactorized, count_params

torch.manual_seed(0)
print("torch", torch.__version__, "| CPU threads:", torch.get_num_threads())

torch 2.15.0+cpu | CPU threads: 10


## 2. The Dataset: how PyTorch gets clips

A PyTorch `Dataset` answers two questions: *how many examples?* (`len`) and *give me example k* (`ds[k]`). Ours is `get_clip` from Phase 2, wrapped in a class.

A `DataLoader` then pulls examples in random order and stacks them into **batches**. The model always sees a batch at once, here 16 clips.

In [3]:
train_ds = ClipDataset(OUT_DIR, "train")
val_ds = ClipDataset(OUT_DIR, "val")
print(f"train: {len(train_ds):,} clips | val: {len(val_ds):,} clips")

x, y = train_ds[0]
print("one clip:", tuple(x.shape), x.dtype, "| label:", int(y), train_ds.class_names[y])

# num_workers=0 is the safe choice in Jupyter on Windows (parallel workers need a .py script there)
loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
xb, yb = next(iter(loader))
print("one batch:", tuple(xb.shape), "labels:", [train_ds.class_names[i] for i in yb[:6]], "...")

train: 70,362 clips | val: 13,206 clips
one clip: (16, 84, 84) torch.uint8 | label: 3 LEFT
one batch: (16, 16, 84, 84) labels: ['UP', 'DOWN', 'RIGHT', 'UP', 'LEFT', 'UP'] ...


Shape `(16, 16, 84, 84)` = 16 clips × 16 frames × 84 × 84 pixels. The first 16 is the **batch**, the second is **time**.

## 3. Follow one batch through the model

`trace=True` prints the shape after every step. Read it with the plan from `src/model.py` next to it:

| Step | Job |
| --- | --- |
| **3D conv** | Looks at each frame *together with its neighbours* (3 frames at a time). Motion only exists across frames, so this is where "Pac-Man moved" first becomes visible. |
| **Fold frames into the batch** | A trick: treat 2 clips × 16 frames as 32 separate images so one 2D CNN processes them all with shared weights. |
| **2D convs ×4** | Each halves the image (84 → 42 → 21 → 11 → 6) while adding channels (64 → 512). Small details turn into higher-level features: walls, Pac-Man, ghosts, direction. |
| **Average-pool** | Each frame becomes one list of 512 numbers. |
| **Transformer** | Each frame's 512 numbers get to *look at all the other frames*: past and future. No mask, so the future is visible. That's the IDM's advantage over a player: it can see what happened next. |
| **Token 8 + head** | Take the labeled frame's features, map 512 → 128 → 9 scores. |

In [4]:
model = IDM9Way()
model.eval()                        # evaluation mode (turns off dropout; see step 6)
with torch.no_grad():
    logits = model(xb[:2], trace=True)

input (uint8)                              (2, 16, 84, 84)
scale to 0-1, add channel                  (2, 1, 16, 84, 84)
3D conv: each frame + neighbours           (2, 32, 16, 84, 84)
fold 16 frames into the batch              (32, 32, 84, 84)
2D conv 1 (stride 2 halves the size)       (32, 64, 42, 42)
2D conv 2 (stride 2 halves the size)       (32, 128, 21, 21)
2D conv 3 (stride 2 halves the size)       (32, 256, 11, 11)
2D conv 4 (stride 2 halves the size)       (32, 512, 6, 6)
average-pool -> one vector per frame       (2, 16, 512)
+ position, transformer (no mask)          (2, 16, 512)
take token 8 (the labeled frame)           (2, 512)
head -> 9 action scores (logits)           (2, 9)


## 4. Where the 7.9 million weights live

In [5]:
parts = {
    "3D conv": model.backbone.temporal,
    "2D convs + batchnorm": [model.backbone.convs, model.backbone.norms],
    "transformer": model.backbone.transformer,
    "head": model.head,
}
total = count_params(model)
for name, mod in parts.items():
    n = sum(count_params(m) for m in (mod if isinstance(mod, list) else [mod]))
    print(f"{name:22s} {n:>10,}  {100 * n / total:5.1f}%")
print(f"{'total':22s} {total:>10,}")
print(f"\nfactorized version (Phase 6): {count_params(IDMFactorized()):,}")

3D conv                       896    0.0%
2D convs + batchnorm    1,569,600   19.8%
transformer             6,304,768   79.4%
head                       66,825    0.8%
total                   7,942,089

factorized version (Phase 6): 8,007,366


The 9-way count should be exactly **7,942,089**, the same as the original repo's log. Same architecture, now with correct labels.

Most of the weights sit in the transformer. The 3D conv is tiny (896 weights) but does the most important job: making motion visible.

## 5. Sanity check 1: the untrained model is a fair coin

**Softmax** turns the 9 scores into 9 probabilities that add up to 1. **Cross-entropy loss** = −log(probability given to the correct action).

An untrained model knows nothing, so it should give each action about 1/9. Then the loss ≈ −log(1/9) = ln 9 ≈ **2.197**. If you see something very different (like 10 or 0.1), something is wrong before training even starts.

In [6]:
with torch.no_grad():
    logits = model(xb)
    probs = F.softmax(logits, dim=1)
    loss = F.cross_entropy(logits, yb)

print("probabilities for clip 0:", np.round(probs[0].numpy(), 3))
print(f"loss = {loss.item():.3f}   (expected about ln 9 = {math.log(9):.3f})")

probabilities for clip 0: [0.106 0.114 0.09  0.126 0.115 0.106 0.108 0.087 0.147]
loss = 2.190   (expected about ln 9 = 2.197)


## 6. How long would real training take on this laptop?

We time a few **training steps**. One step = show a batch, measure the loss, compute how each weight should change (`backward`), change them (`optimizer.step`).

`model.train()` switches **dropout** on: during training it randomly zeroes 10% of the head's numbers so the model can't lean on any single one. Eval mode turns it off so predictions are stable.

In [7]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
t0 = time.time()
for _ in range(3):
    opt.zero_grad()
    loss = F.cross_entropy(model(xb), yb)
    loss.backward()
    opt.step()
sec = (time.time() - t0) / 3
steps = len(train_ds) // 16
print(f"{sec:.2f} s per step  ->  one pass over train ({steps:,} steps) ≈ {sec * steps / 3600:.1f} hours on this CPU")
print("15 passes (what we'll do in Phase 4) ≈", f"{15 * sec * steps / 3600:.0f} hours. A Colab GPU does it in about an hour.")

11.55 s per step  ->  one pass over train (4,397 steps) ≈ 14.1 hours on this CPU
15 passes (what we'll do in Phase 4) ≈ 212 hours. A Colab GPU does it in about an hour.


That number is why Phase 4 moves to Colab.

## 7. Sanity check 2: memorize one batch

Take **18 clips, 2 of each action**, and train on only those, over and over. A working model should drive the loss toward 0 and get all 18 right, because it can simply memorize them.

- **It does:** the model, the labels and the training loop are wired correctly.
- **It doesn't:** there's a bug (wrong labels, wrong shapes, a broken layer). Real training would have failed too, just more slowly and more confusingly.

This takes a few minutes on a laptop (usually 15–40 steps). Watch the loss fall.

In [8]:
torch.manual_seed(0)
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.flatnonzero(train_ds.labels == c), 2, replace=False) for c in range(9)])
xs = torch.stack([train_ds[i][0] for i in idx])
ys = torch.stack([train_ds[i][1] for i in idx])
print("batch:", [train_ds.class_names[i] for i in ys])

model = IDM9Way()                    # fresh, untrained
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = []
t0 = time.time()
for step in range(1, 201):
    model.train()
    opt.zero_grad()
    logits = model(xs)
    loss = F.cross_entropy(logits, ys)
    loss.backward()
    opt.step()
    history.append(loss.item())
    correct = (logits.argmax(1) == ys).sum().item()
    if step % 5 == 0:
        print(f"step {step:3d}  loss {loss.item():.3f}  correct {correct}/18  ({time.time() - t0:.0f}s)")
    if correct == 18 and loss.item() < 0.05:
        print(f"Memorized at step {step}. The model can learn.")
        break

batch: ['UPLEFT', 'UPLEFT', 'UP', 'UP', 'UPRIGHT', 'UPRIGHT', 'LEFT', 'LEFT', 'NOOP', 'NOOP', 'RIGHT', 'RIGHT', 'DOWNLEFT', 'DOWNLEFT', 'DOWN', 'DOWN', 'DOWNRIGHT', 'DOWNRIGHT']
step   5  loss 2.007  correct 3/18  (34s)
step  10  loss 1.598  correct 8/18  (57s)
step  15  loss 0.961  correct 13/18  (79s)
step  20  loss 0.435  correct 16/18  (101s)
step  25  loss 0.134  correct 17/18  (122s)
Memorized at step 28. The model can learn.


In [ ]:
plt.figure(figsize=(7, 3.2))
plt.plot(history)
plt.axhline(math.log(9), ls="--", c="gray"); plt.text(1, math.log(9) + 0.05, "random guessing (ln 9)", color="gray")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Memorizing 18 clips")
plt.tight_layout(); plt.show()

### A gotcha worth knowing: train mode vs eval mode

Now check the same 18 clips in **eval mode**, which is how predictions are made for real:

In [9]:
model.eval()
with torch.no_grad():
    print("correct in eval mode:", (model(xs).argmax(1) == ys).sum().item(), "/ 18")

correct in eval mode: 2 / 18


This can come out **much lower than 18**, and it isn't a bug. It's **BatchNorm**, the `bn` layers after each 2D conv.

- In **train mode**, BatchNorm rescales each layer's numbers using the current batch's own average and spread.
- In **eval mode**, it uses a *running average* of those values, collected slowly during training (each step moves it only 10% of the way).

Here the weights changed drastically in ~20 steps, so the running averages are still far behind and eval-mode predictions are off. In real training (Phase 4) there are thousands of steps with small updates, and the running averages keep up.

Write this one down. Train/eval mismatches are one of the most common silent bugs in deep learning, and it makes a good short section in the article.

**What you should see above:** loss starts near 2.2, drops fast, and ends near 0 with 18/18 correct in train mode.

**Important:** memorizing 18 clips says nothing about how well the model works on *new* clips. That's the real question, and Phases 4–5 answer it with val and test. This check only proves the machinery works.

## Done with Phase 3

**For `notes.md`:** your loss at step 5, your seconds per step and the hours estimate from step 6, how many steps it took to memorize the batch, and the train-vs-eval mode gap.

**Next, Phase 4:** the same model on all 70k training clips, on a Colab GPU, checking val after every pass. You'll upload `frames.npy` + `splits.npz` + `meta.json` (≈ 2.7 GB) to Google Drive once and train from there.